# EDA — Explications du Conseil fédéral

Ce notebook présente l'analyse exploratoire du corpus **Chancellerie fédérale – Explications du Conseil fédéral** disponible dans les exports courants (couverture depuis 1978).

Il ne réimplémente pas la logique de l'EDA : il réutilise `scripts/federal_explanations/federal_explanations_eda.py`, afin de garder une seule source de vérité testable et reproductible.


## 1. Chargement des exports du pipeline

Le notebook lit les données générées par le pipeline : `objects.parquet`, `sections.parquet` et `pages.parquet`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "environment.yml").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Repository root not found")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.federal_explanations.federal_explanations_eda import generate_eda

INPUT_DIR = ROOT / "data" / "processed" / "federal_explanations"
OUTPUT_DIR = ROOT / "reports" / "generated" / "federal_explanations" / "eda"

objects = pd.read_parquet(INPUT_DIR / "objects.parquet")
sections = pd.read_parquet(INPUT_DIR / "sections.parquet")
pages = pd.read_parquet(INPUT_DIR / "pages.parquet")

print(f"Objects:       {len(objects)}")
print(f"Sections:      {len(sections)}")
print(f"Pages:         {len(pages)}")


## 2. Aperçu de la couverture

Quelques contrôles simples permettent de vérifier la période, la couverture et les statuts avant d'interpréter les visualisations.

In [ ]:
coverage_preview = objects[[
    column for column in [
        "vote_id", "vote_date", "vote_title_fr", "brochure_status",
        "availability_status", "semantic_status", "exclusion_reasons"
    ] if column in objects.columns
]].copy()

display(coverage_preview.head(10))


## 3. Génération de l'EDA

Le script génère trois figures et un rapport Markdown à partir des artefacts courants.

In [ ]:
generated = generate_eda(INPUT_DIR, OUTPUT_DIR)
generated


## 4. Couverture des objets par année

Ce graphique montre le nombre d'objets couverts chaque année et permet de repérer rapidement une éventuelle rupture de couverture.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "coverage_by_year.png")))

## 5. Qualité de l'extraction

La majorité des pages doit être extraite directement depuis la couche texte du PDF. L'OCR n'est utilisé qu'en fallback.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "extraction_quality.png")))

## 6. Longueur des textes extraits

Cette distribution permet de contrôler l'hétérogénéité de la quantité de texte disponible par objet et de repérer d'éventuelles extractions anormalement courtes ou longues.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "text_length_distribution.png")))

## 7. Synthèse automatique

Le rapport ci-dessous reprend les principaux compteurs de couverture, extraction et éligibilité au modèle.

In [ ]:
report = (OUTPUT_DIR / "report.md").read_text(encoding="utf-8")
display(Markdown(report))

## Interprétation

- La couverture et l'extraction technique doivent être vérifiées avant toute utilisation comme feature ML.
- Un PDF correctement extrait n'est pas automatiquement **éligible au modèle** : la validation sémantique et la disponibilité pré-scrutin restent des critères séparés.
- Cette EDA sert principalement au **contrôle qualité du dataset Chancellerie**. L'EDA politique principale sera réalisée après fusion avec les résultats communaux et les autres features.